In [ ]:
from openai import OpenAI
client = OpenAI(
    api_key='sk-de1c0a133b224e309f7ba2415b9764ba',
    base_url='https://api.deepseek.com/v1'
)

In [ ]:
import sqlite3

conn = sqlite3.connect('test.db')

cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS employees(
    id INTEGER PRIMARY KEY,
    name TEXT,
    deparment TEXT,
    salary INTEGER
)
""")

sample_data = [
    (6,'黄家','销售',50000),
    (7,'宁宁','工程',75000),
    (8,'钱钱','销售',60000),
    (9,'月月','工程',80000),
    (10,'黄仁勋','市场',55000),
    (11,'户撒大噶','工程',80000)
]

cursor.executemany("""
INSERT INTO employees
VALUES(?,?,?,?)
""", sample_data)

conn.commit()

In [ ]:
schema = cursor.execute("""PRAGMA table_info(employees)""").fetchall()
print(schema)
# 列表推导式
schema_str = "CREATE TABLE EMPLOYEES(\n" + "\n".join([f"{col[1]} {col[2]}" for col in schema]) + "\n)"
print(schema_str)

In [ ]:
messages=[]
def ask_deepseek(query,msg,schema):
    prompt = f"""
    这是一个数据库的schema:{schema},
    根据这个schema，请输出一个sql查询来回答以下问题.
    只输出sql查询语句本身，不要使用任何Markdown格式语句,
    不要包含反引号、代码块标记或 额外说明.
    问题:{query}
    """
    msg.append({
        "role":"user",
        "content":prompt
    })
    response = client.chat.completions.create(
        model='deepseek-v4-flash',
        max_tokens=2048,
        messages=msg,
        temperature=0
    )
    msg.append({
        "role":"assistant",
        "content":response.choices[0].message.content
    })
    return response.choices[0].message.content
question = "工程部门的员工的姓名和工资是多少"
sql_query = ask_deepseek(question,messages,schema_str)

In [ ]:
results = cursor.execute(sql_query).fetchall()
print(results)

[('宁宁', 75000), ('月月', 80000), ('户撒大噶', 80000)]


In [ ]:
question = "在销售部分增加一个新员工，姓名为四名，工资为45000"
sql_query = ask_deepseek(question,messages,schema_str)
cursor.execute(sql_query)
conn.commit()
table_data = cursor.execute("""SELECT * FROM employees""").fetchall()
print(table_data)

[(6, '黄家', '销售', 50000), (7, '宁宁', '工程', 75000), (8, '钱钱', '销售', 60000), (9, '月月', '工程', 80000), (10, '黄仁勋', '市场', 55000), (11, '户撒大噶', '工程', 80000), (12, '四名', '销售', 45000), (13, '四名', '销售', 45000), (14, '四名', '销售', 45000), (15, '四名', '销售', 45000)]


In [ ]:
question = "请把所有名字为四名的销售全部删去，除了四名中id最小的那个四名"
sql_query = ask_deepseek(question,messages,schema_str)
cursor.execute(sql_query)
conn.commit()

In [ ]:
table_data = cursor.execute("""SELECT * FROM employees""").fetchall()
print(table_data)

[(6, '黄家', '销售', 50000), (7, '宁宁', '工程', 75000), (8, '钱钱', '销售', 60000), (9, '月月', '工程', 80000), (10, '黄仁勋', '市场', 55000), (11, '户撒大噶', '工程', 80000), (12, '四名', '销售', 45000)]


In [ ]:
question = "删除市场部门的黄仁勋"
sql_query = ask_deepseek(question,messages,schema_str)
cursor.execute(sql_query)
conn.commit()

In [ ]:
table_data = cursor.execute("""SELECT * FROM employees""").fetchall()
print(table_data)

[(6, '黄家', '销售', 50000), (7, '宁宁', '工程', 75000), (8, '钱钱', '销售', 60000), (9, '月月', '工程', 80000), (11, '户撒大噶', '工程', 80000), (12, '四名', '销售', 45000)]
